# R03 · Benign reconstruction transfer (Sections 4.3–4.5; data of Figs. 5–7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R03_Benign_Transfer_Analysis.ipynb)

Recomputes the exceedances of the smoke gates, the false-positive rates by multiplicity band (row and group level), the 00C/00D diagnostics and the quantile ladder.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Reproduce the benign reconstruction-transfer results (manuscript Sections 4.3-4.5; Figs. 5-7 data).

Inputs (all inside the repository):
  artifacts/unsw_nb15/partition/source_balanced/{smoke_manifest_source_balanced.json,
      development_benign_fingerprint_level_summary.csv, diagnostic_manifest_00C_*.json, diagnostic_manifest_00D_*.json}
  artifacts/unsw_nb15/partition/final_source_multiplicity_balanced/{smoke_manifest_final.json, benign_quantile_ladder.csv}
Outputs:
  results/figure_source_data/Figure_3_2_data.csv   development-benign exceedance by regime (manuscript Fig. 6)
  results/figure_source_data/Figure_3_3_data.csv   false-positive rate by multiplicity band (manuscript Fig. 5)
  results/figure_source_data/Figure_3_4_data.csv   quantile ladder and ratios (manuscript Fig. 7)
  results/claims_check/reproduce_transfer_analysis.csv
Two plotted values have no artefact and are carried, labelled, from the analysis record: the
initial-allocation exceedance (99.22%) and the CICIDS2017 retrospective quantile ratios.

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

In [ ]:
# ===== src/utils/claims.py =====
"""Record manuscript claims next to the values recomputed from the artefacts.

Each check compares a recomputed number with the value *as printed in the manuscript*. The tolerance is
half a unit in the last printed decimal place, so "0.7094" passes for any value in [0.70935, 0.70945].
Every reproduction script writes its log to ``results/claims_check/<script>.csv`` and exits with a
non-zero status if any check fails.
"""

import math
import sys
from pathlib import Path

import pandas as pd



def _parse(printed: str) -> tuple[float, float]:
    s = str(printed).replace(",", "").replace("%", "").replace("+", "").replace("−", "-").strip()
    dec = len(s.split(".")[1]) if "." in s and "e" not in s.lower() else 0
    if "e" in s.lower():
        mant = s.lower().split("e")[0]
        exp = int(s.lower().split("e")[1])
        dec_m = len(mant.split(".")[1]) if "." in mant else 0
        return float(s), 0.5 * 10 ** (exp - dec_m)
    return float(s), 0.5 * 10 ** (-dec)


class ClaimLog:
    def __init__(self, script: str):
        self.script = script
        self.rows: list[dict] = []

    def check(self, claim_id: str, location: str, description: str, reproduced, printed: str,
              source: str, scale: float = 1.0) -> bool:
        """``scale`` converts the reproduced value to the printed unit (e.g. 100 for percentages)."""
        value, tol = _parse(printed)
        rep = float(reproduced) * scale
        ok = math.isclose(rep, value, abs_tol=tol + 1e-12)
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": printed, "reproduced_value": repr(round(rep, 10)),
                          "status": "PASS" if ok else "FAIL", "source": source})
        return ok

    def assert_true(self, claim_id: str, location: str, description: str, condition: bool, source: str) -> bool:
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "true", "reproduced_value": str(bool(condition)),
                          "status": "PASS" if condition else "FAIL", "source": source})
        return bool(condition)

    def note(self, claim_id: str, location: str, description: str, value: str, source: str, status: str) -> None:
        """Record a finding that is not a pass/fail comparison (e.g. a discrepancy that the manuscript notes)."""
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "", "reproduced_value": value, "status": status, "source": source})

    def finish(self) -> int:
        df = pd.DataFrame(self.rows)
        out = RESULTS / "claims_check" / f"{self.script}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out, index=False, lineterminator="\n")
        n_fail = int((df.status == "FAIL").sum())
        n_pass = int((df.status == "PASS").sum())
        print(f"[{self.script}] {n_pass} checks passed, {n_fail} failed, {len(df) - n_pass - n_fail} notes -> {out}")
        if n_fail:
            print(df[df.status == "FAIL"].to_string(index=False), file=sys.stderr)
        return 1 if n_fail else 0

In [ ]:
# ===== src/fingerprinting/multiplicity_audit.py =====
"""Fingerprint-multiplicity bands, total-variation distance and related structural quantities.

The eight frozen bands are those of the UNSW-NB15 source-multiplicity amendment
(``source_multiplicity_amendment.json["bands"]``): 1, 2, 3-4, 5-9, 10-19, 20-49, 50-99, >=100.
"""

import numpy as np
import pandas as pd

BANDS = ["1", "2", "3-4", "5-9", "10-19", "20-49", "50-99", ">=100"]
EDGES = [0, 1, 2, 4, 9, 19, 49, 99, np.inf]   # right-closed intervals (0,1], (1,2], (2,4], ...


def band_of(group_size) -> pd.Series:
    """Map fingerprint-group sizes (multiplicities) to the frozen band labels."""
    return pd.cut(pd.Series(group_size), bins=EDGES, labels=BANDS, right=True).astype(str)


def shares(rows_by_band: pd.Series) -> pd.Series:
    """Band shares of rows, in the frozen band order."""
    r = rows_by_band.reindex(BANDS).fillna(0).astype(float)
    return r / r.sum()


def tv_distance(p: pd.Series, q: pd.Series) -> float:
    """Total-variation distance 1/2 * sum_b |p_b - q_b| between two band-share vectors."""
    return float(0.5 * (shares(p) - shares(q)).abs().sum())


def gate_rows(gate: pd.DataFrame, column: str = "actual_rows") -> pd.DataFrame:
    """Role x band row matrix from ``multiplicity_gate.csv``."""
    return gate.pivot(index="partition", columns="size_band", values=column)[BANDS]


def expected_cross_side_sharing(group_sizes: np.ndarray) -> float:
    """Expected number of rows sharing their fingerprint with >=1 row on the opposite side when every row
    is assigned independently to one of two sides with probability 1/2: sum_g m_g (1 - 2^{-(m_g - 1)})."""
    m = np.asarray(group_sizes, dtype=float)
    return float((m * (1.0 - np.power(0.5, m - 1.0))).sum())


def redundancy_rate(rows: int, groups: int) -> float:
    """(rows - groups) / rows: the share of rows that repeat a model-view vector already present."""
    return (rows - groups) / rows

## Reproduction: `scripts/reproduce_transfer_analysis.py`

In [ ]:
import sys
from pathlib import Path


import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402


L = ClaimLog("reproduce_transfer_analysis")
SB = ("unsw_nb15", "partition", "source_balanced")
F = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")
REC = "analysis record (no artefact in the evidence set)"
INITIAL_EXCEEDANCE_RECORD = 99.22
CIC_RETRO_RATIOS_RECORD = [1.090, 0.959, 0.974, 0.989, 0.984, 0.984, 0.937, 0.945]

sb = load_json(art(*SB, "smoke_manifest_source_balanced.json"))
fn = load_json(art(*F, "smoke_manifest_final.json"))
d00c = load_json(art(*SB, "diagnostic_manifest_00C_block_decomposition.json"))
d00d = load_json(art(*SB, "diagnostic_manifest_00D_mixed_label.json"))
lad = read_csv(art(*F, "benign_quantile_ladder.csv"))
fs = read_csv(art(*SB, "development_benign_fingerprint_level_summary.csv"))

### Section 4.3: source-balanced regime

In [ ]:
L.assert_true("S4.3.status", "Sec. 4.3", "source-balanced smoke gate FAIL; v3.4 smoke gate PASS",
              sb["final_status"] == "FAIL" and fn["final_status"] == "PASS", "smoke manifests")
L.check("S4.3.dev_exc", "Sec. 4.3; Table 8; abstract", "development-benign exceedance, source-balanced (%)", 100 * sb["development_benign_exceedance"], "53.63", "smoke_manifest_source_balanced.json")
L.check("S4.3.dev_exc_frac", "Sec. 4.3", "development-benign exceedance, source-balanced (fraction)", sb["development_benign_exceedance"], "0.536263", "as above")
L.check("S4.3.cal_exc", "Sec. 4.3; Table 8", "calibration-benign exceedance, source-balanced (%)", 100 * sb["calibration_benign_exceedance"], "1.0003", "as above")
L.check("S4.3.dev_rows", "Sec. 4.3", "development-benign rows", d00d["development_benign_rows"], "221,876", "diagnostic_manifest_00D_mixed_label.json")
L.check("S4.3.dev_rows_summary", "Sec. 4.3", "development-benign rows in the fingerprint-level summary", fs.benign_rows.sum(), "221,876", "development_benign_fingerprint_level_summary.csv")
L.check("S4.3.reduction_pp", "Sec. 4.3", "reduction from the initial regime (percentage points; uses the 99.22% record)",
        INITIAL_EXCEEDANCE_RECORD - 100 * sb["development_benign_exceedance"], "45.59", "smoke manifest + " + REC)
L.assert_true("S4.3.block", "Sec. 4.3", "numerical block is the leading contributor to the error increase",
              d00c["leading_shift_block_by_absolute_mean_total_mse_contribution_increase"] == "numerical", "diagnostic_manifest_00C_block_decomposition.json")
L.check("S4.3.pure_rows", "Sec. 4.3", "development-benign rows in pure-benign fingerprints", d00d["development_benign_pure_fingerprint_rows"], "220,890", "00D manifest")
L.check("S4.3.mixed_rows", "Sec. 4.3", "development-benign rows in mixed-label fingerprints", d00d["development_benign_mixed_fingerprint_rows"], "986", "00D manifest")
for k, p in [("calibration_ct7_pooled_median_abs_z", "0.479"), ("development_mixed_ct7_pooled_median_abs_z", "0.657"),
             ("development_pure_ct7_pooled_median_abs_z", "4.471")]:
    L.check(f"S4.3.{k}", "Sec. 4.3", k.replace("_", " "), d00d[k], p, "00D manifest")
L.assert_true("S4.3.ct7_features", "Sec. 4.3", "seven connection-count features", len(d00d["ct_density_features"]) == 7, "00D manifest")
L.assert_true("S4.3.threshold_consistent", "Sec. 4.3", "00D threshold recomputation agrees with the source-balanced smoke threshold",
              abs(d00d["threshold_value_recomputed_from_00c_calibration_benign"] - sb["threshold"]) < 1e-6, "00D manifest + smoke manifest")

### Section 4.4: multiplicity bands (Fig. 5)

In [ ]:
L.assert_true("S4.4.group_level_binary", "Sec. 4.4", "every fingerprint group lies entirely above or entirely below the threshold",
              bool(fs.above_calibration_p99_rate.isin([0.0, 1.0]).all()), "development_benign_fingerprint_level_summary.csv")
mix = fs[fs.fingerprint_group_class != "pure benign group"]
L.check("S4.3.mixed_fp", "Sec. 4.3", "mixed-label benign rows above the threshold", (mix.benign_rows * mix.above_calibration_p99_rate).sum(), "6", "as above")
p = fs[fs.fingerprint_group_class == "pure benign group"].copy()
p["band"] = band_of(p.benign_rows).to_numpy()
p["fp_rows"] = p.benign_rows * p.above_calibration_p99_rate
g = p.groupby("band").agg(rows=("benign_rows", "sum"), groups=("benign_rows", "size"), fp_rows=("fp_rows", "sum"),
                          fp_groups=("above_calibration_p99_rate", "sum")).reindex(BANDS)
g["row_rate_pct"] = 100 * g.fp_rows / g.rows
g["group_rate_pct"] = 100 * g.fp_groups / g.groups
L.check("S4.3.pure_fp", "Sec. 4.3", "pure-benign development rows above the threshold", g.fp_rows.sum(), "118,978", "as above")
L.check("S4.3.pure_fp_pct", "Sec. 4.3", "share of pure-benign development rows above the threshold", 100 * g.fp_rows.sum() / g.rows.sum(), "53.86", "as above")
for b_, pr in zip(BANDS, ["0.013", "8.45", "6.42", "55.12", "77.34", "92.36", "96.66", "98.83"]):
    L.check(f"S4.4.row_rate.{b_}", "Sec. 4.4; Fig. 5", f"row-level FP rate, band {b_} (%)", g.loc[b_, "row_rate_pct"], pr, "as above")
for b_, pr in zip(BANDS, ["0.013", "8.45", "6.23", "50.74", "77.51", "91.16", "96.27", "98.20"]):
    L.check(f"S4.4.group_rate.{b_}", "Sec. 4.4", f"group-level FP rate, band {b_} (%)", g.loc[b_, "group_rate_pct"], pr, "as above")
L.check("S4.4.singleton_fp", "Sec. 4.4", "singleton rows above threshold", g.loc["1", "fp_rows"], "4", "as above")
L.check("S4.4.singleton_rows", "Sec. 4.4", "singleton rows", g.loc["1", "rows"], "30,766", "as above")
L.check("S4.4.ge100_fp", "Sec. 4.4", ">=100 band rows above threshold", g.loc[">=100", "fp_rows"], "38,743", "as above")
L.check("S4.4.ge100_rows", "Sec. 4.4", ">=100 band rows", g.loc[">=100", "rows"], "39,202", "as above")
L.check("abstract.ge100_rate", "Abstract", ">=100 band false-positive rate (%)", g.loc[">=100", "row_rate_pct"], "98.8", "as above")
L.check("highlights.range_hi", "Highlights", ">=100 band false-positive rate (%)", g.loc[">=100", "row_rate_pct"], "98.83", "as above")
hi = g.loc[["10-19", "20-49", "50-99", ">=100"]]
L.check("S4.4.ge10_rows_share", "Sec. 4.4", "rows in groups of >=10 (% of pure-benign development rows)", 100 * hi.rows.sum() / g.rows.sum(), "51.2", "as above")
L.check("S4.4.ge10_fp_share", "Sec. 4.4", "share of false positives in groups of >=10 (%)", 100 * hi.fp_rows.sum() / g.fp_rows.sum(), "90.4", "as above")
L.assert_true("S4.4.monotone_from_3", "Sec. 4.4", "row rate monotone from band 3-4 upward, but band 2 exceeds band 3-4",
              bool(np.all(np.diff(g.row_rate_pct.to_numpy()[2:]) > 0)) and g.loc["2", "row_rate_pct"] > g.loc["3-4", "row_rate_pct"], "as above")

### Section 4.5: v3.4 regime and ladder (Fig. 7)

In [ ]:
L.check("S4.5.dev_exc", "Sec. 4.5; Table 8; abstract", "development-benign exceedance, v3.4 (%)", 100 * fn["development_benign_exceedance"], "1.0024", "smoke_manifest_final.json")
L.check("S4.5.cal_exc", "Sec. 4.5; Table 8", "calibration-benign exceedance, v3.4 (%)", 100 * fn["calibration_benign_exceedance"], "1.0003", "smoke_manifest_final.json")
L.check("S4.5.ratio", "Sec. 4.5", "development/calibration exceedance ratio, v3.4", fn["development_to_calibration_exceedance_ratio"], "1.0021", "smoke_manifest_final.json")
for q, pr in zip(lad["quantile"], ["1.005", "1.022", "1.031", "1.047", "0.984", "0.960", "1.006", "0.989"]):
    L.check(f"S4.5.ladder.{q:g}", "Sec. 4.5; Fig. 7b", f"development/calibration quantile ratio at {q:g}",
            float(lad.loc[lad["quantile"] == q, "dev_to_cal_ratio"].iloc[0]), pr, "benign_quantile_ladder.csv")
L.assert_true("S4.5.ladder_within_5pct", "Sec. 4.5", "all eight ratios within 5% of unity and inside the frozen 0.2-5.0 range",
              bool(((lad.dev_to_cal_ratio - 1).abs() < 0.05).all()) and bool(lad.ratio_guardrail_pass.all())
              and fn["guardrails"]["quantile_scale_ratio_range"] == [0.2, 5.0], "benign_quantile_ladder.csv + smoke_manifest_final.json")
L.assert_true("S4.5.threshold_is_q99", "Methods (threshold)", "the gate threshold equals the calibration-benign 0.99 quantile of the ladder (to 12 significant digits)",
              np.isclose(float(lad.loc[lad["quantile"] == 0.99, "calibration_benign"].iloc[0]), fn["threshold"], rtol=1e-12, atol=0), "benign_quantile_ladder.csv + smoke_manifest_final.json")
L.note("S4.6.cic_retro", "Sec. 4.6; Fig. 7b", "CICIDS2017 retrospective check (1.0001%, 0.9186%, eight ratios)",
       "not recomputable: no calibration/development score artefact retained for the search-stage AE", REC, "NOT_REPRODUCIBLE_HERE")

### outputs

In [ ]:
write_csv(pd.DataFrame([
    {"regime": "Initial atomic", "development_benign_exceedance_pct": INITIAL_EXCEEDANCE_RECORD, "calibration_benign_exceedance_pct": np.nan, "source": REC},
    {"regime": "Source-balanced", "development_benign_exceedance_pct": 100 * sb["development_benign_exceedance"],
     "calibration_benign_exceedance_pct": 100 * sb["calibration_benign_exceedance"], "source": "smoke_manifest_source_balanced.json"},
    {"regime": "Source- and multiplicity-balanced (v3.4)", "development_benign_exceedance_pct": 100 * fn["development_benign_exceedance"],
     "calibration_benign_exceedance_pct": 100 * fn["calibration_benign_exceedance"], "source": "smoke_manifest_final.json"}]),
    RESULTS / "figure_source_data" / "Figure_3_2_data.csv")
write_csv(g.reset_index().rename(columns={"index": "band"}), RESULTS / "figure_source_data" / "Figure_3_3_data.csv")
lad_out = lad.copy()
lad_out["cicids2017_retrospective_ratio_analysis_record"] = CIC_RETRO_RATIOS_RECORD
write_csv(lad_out, RESULTS / "figure_source_data" / "Figure_3_4_data.csv")
assert L.finish() == 0, "some checks FAILED - see the claim log below"

## Results

In [ ]:
# ===== Results of the checks =====
import pandas as pd
from IPython.display import display
log = pd.read_csv(REPO_ROOT / "results" / "claims_check" / "reproduce_transfer_analysis.csv")
print(log.status.value_counts().to_string())
display(log[log.status != "PASS"])          # notes: values that cannot be recomputed here, or corrections
display(log.head(15))                        # first passing checks (all rows are in the CSV)

In [ ]:
# ===== Tables written by this notebook =====
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 30)
print("results/figure_source_data/Figure_3_2_data.csv"); display(pd.read_csv(REPO_ROOT / "results/figure_source_data/Figure_3_2_data.csv"))
print("results/figure_source_data/Figure_3_3_data.csv"); display(pd.read_csv(REPO_ROOT / "results/figure_source_data/Figure_3_3_data.csv"))
print("results/figure_source_data/Figure_3_4_data.csv"); display(pd.read_csv(REPO_ROOT / "results/figure_source_data/Figure_3_4_data.csv"))